# 🍽️ ChefMind RAG System
**Hybrid Retrieval-Augmented Generation for Recipe Q&A**

| Step | Component |
|------|----------|
| 1 | Install packages |
| 2 | Load indexes from Drive |
| 3 | Retrieval functions |
| 4 | Cross-Encoder reranker |
| 5 | Load ChefMind LoRA model |
| 5b | Name lookup from 3 datasets |
| 6 | RAG pipeline |
| 7 | Expose via ngrok API |

## 📦 Step 1: Install Packages

In [ ]:
%%capture
!pip install datasets langchain langchain-community langchain-core \
             langchain-text-splitters langchain-huggingface \
             faiss-cpu rank-bm25 \
             sentence-transformers pyngrok nest-asyncio fastapi uvicorn \
             transformers peft accelerate bitsandbytes -q
print("✅ Packages installed!")

## ♻️ Step 2: Load Indexes from Drive
*Skip Steps 2–3 if you just built them.*

In [ ]:
import pickle
from google.colab import drive
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from rank_bm25 import BM25Okapi

drive.mount("/content/drive")

drive_dir = "/content/drive/MyDrive/ChefMind/indexes"

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.load_local(f"{drive_dir}/faiss_index", embeddings, allow_dangerous_deserialization=True)
print("✅ FAISS loaded!")

with open(f"{drive_dir}/bm25.pkl", "rb") as f:
    bm25 = pickle.load(f)
with open(f"{drive_dir}/chunks.pkl", "rb") as f:
    chunks = pickle.load(f)
print(f"✅ BM25 + {len(chunks)} chunks loaded!")

## 🔀 Step 3: Retrieval Functions

In [ ]:
def generate_queries(q):
    return [q, f"recipe for {q}", f"how to cook {q}", f"ingredients for {q}"]

def bm25_retrieve(query, k=5):
    scores = bm25.get_scores(query.lower().split())
    top_idx = scores.argsort()[-k:][::-1]
    return [(chunks[i], scores[i]) for i in top_idx]

def rrf_fusion(faiss_docs, bm25_results, k=60):
    scores = {}
    for rank, doc in enumerate(faiss_docs):
        key = doc.page_content[:100]
        scores[key] = scores.get(key, 0) + 1/(k + rank + 1)
    for rank, (doc, _) in enumerate(bm25_results):
        key = doc.page_content[:100]
        scores[key] = scores.get(key, 0) + 1/(k + rank + 1)
    all_docs = {d.page_content[:100]: d for d in faiss_docs}
    all_docs.update({d.page_content[:100]: d for d, _ in bm25_results})
    return [all_docs[k] for k, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True) if k in all_docs]

def hybrid_retrieve(question, k=5):
    return rrf_fusion(vectorstore.similarity_search(question, k=k), bm25_retrieve(question, k=k))

print("✅ Retrieval functions ready!")

## 🤖 Step 4: Cross-Encoder Reranker

In [ ]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, docs, top_k=5):
    if not docs:
        return docs
    pairs = [[query, doc.page_content] for doc in docs]
    scores = cross_encoder.predict(pairs)
    return [doc for _, doc in sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)[:top_k]]

print("✅ Cross-Encoder ready!")

## 🧠 Step 5: Load ChefMind LoRA Model
*Takes ~3 minutes*

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct"
LORA_REPO  = "Osamaelbably87/chefmind-lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("⏳ Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(LORA_REPO)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"⏳ Loading base model {BASE_MODEL} in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"⏳ Applying LoRA adapter from {LORA_REPO}...")
model = PeftModel.from_pretrained(base_model, LORA_REPO)
model.eval()
print("✅ Model loaded!")

## 🔤 Step 5b: Recipe Name Lookup
Builds a name index from all 3 datasets so every recipe gets a proper name.

In [ ]:
import re
from datasets import load_dataset

print("⏳ Building name lookup from all 3 datasets...")
all_names = []

try:
    ds1 = load_dataset("tengomucho/simple_recipes", split="train")
    all_names += [r['names'] for r in ds1 if r.get('names','').strip()]
    print(f"✅ ds1: {len(all_names)} names")
except Exception as e:
    print(f"⚠️ ds1 failed: {e}")

try:
    ds2 = load_dataset("rahul7star/food-recipes", split="train")
    cols = ds2.column_names
    name_col = next((c for c in ['title','name','names','recipe_name'] if c in cols), cols[0])
    before = len(all_names)
    all_names += [r[name_col] for r in ds2 if str(r.get(name_col,'')).strip()]
    print(f"✅ ds2: {len(all_names)-before} names (column='{name_col}')")
except Exception as e:
    print(f"⚠️ ds2 failed: {e}")

try:
    ds3 = load_dataset("formido/recipes", split="train")
    cols = ds3.column_names
    name_col = next((c for c in ['title','name','names','recipe_name'] if c in cols), cols[0])
    before = len(all_names)
    all_names += [r[name_col] for r in ds3 if str(r.get(name_col,'')).strip()]
    print(f"✅ ds3: {len(all_names)-before} names (column='{name_col}')")
except Exception as e:
    print(f"⚠️ ds3 failed: {e}")

print(f"\n✅ Total recipe names: {len(all_names)}")

def find_best_name(query):
    stopwords = {'and','or','with','the','a','an','in','of','for','to','my','i'}
    q_words = set(re.findall(r'\b\w+\b', query.lower())) - stopwords
    best_name, best_score = None, 0
    for name in all_names:
        if not name or not str(name).strip():
            continue
        n_words = set(re.findall(r'\b\w+\b', str(name).lower())) - stopwords
        score = len(q_words & n_words)
        if score > best_score:
            best_score = score
            best_name = str(name).strip().title()
    if best_name and best_score >= 1:
        return best_name
    parts = [w.strip().title() for w in query.split(',')[:3] if len(w.strip()) > 2]
    return ', '.join(parts) + ' Recipe' if parts else query.title()

print("✅ Name lookup ready!")


## 🍳 Step 6: RAG Pipeline

In [ ]:
import json, re, torch

def run_chefmind_pipeline(query):
    retrieved_docs = hybrid_retrieve(query, k=10)
    reranked_docs = rerank(query, retrieved_docs, top_k=3)

    prompt = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        "You are ChefMind, a creative AI chef. When given a dish name or ingredients, "
        "you provide a complete, delicious recipe with clear steps."
        "<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
        "Give me a recipe for: " + query +
        "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    raw = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    lines = [re.sub(r'^[-•*]\s*', '', l).strip() for l in raw.split('\n') if l.strip()]
    lines = [re.sub(r'\s+\d+\.\s*$', '', l).strip() for l in lines]
    lines = [l for l in lines if l]
    instructions = [f"Step {i+1}: {l}" for i, l in enumerate(lines)]

    time_match = re.search(r'(\d[\d\s\-–]*(?:to\s*\d+)?\s*min)', raw, re.IGNORECASE)
    cooking_time = time_match.group(1).strip() if time_match else "30 minutes"

    ing_list = [{"name": w.strip(), "quantity": ""} for w in query.split(',') if len(w.strip()) > 2]

    return {
        "recipe_name": find_best_name(query),
        "ingredients": ing_list,
        "instructions": instructions if instructions else ["Prepare all ingredients and cook until done."],
        "cooking_time": cooking_time,
        "difficulty": "Medium",
    }

print("✅ ChefMind Pipeline ready!")


## 🌐 Step 7: Expose via ngrok API

In [ ]:
from pyngrok import ngrok

NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"
ngrok.set_auth_token(NGROK_TOKEN)
print("✅ NGROK authenticated!")

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import nest_asyncio, uvicorn, threading
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI(title="ChefMind API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class RecipeRequest(BaseModel):
    query: str

@app.get("/")
def root():
    return {"message": "ChefMind API is running! 🍳"}

@app.post("/recipe")
def get_recipe(request: RecipeRequest):
    result = run_chefmind_pipeline(request.query)
    return {
        "recipe_name":  result.get("recipe_name", ""),
        "ingredients":  result.get("ingredients", []),
        "instructions": result.get("instructions", []),
        "cooking_time": result.get("cooking_time", ""),
        "difficulty":   result.get("difficulty", ""),
    }

ngrok_tunnel = ngrok.connect(8000)
print(f"🌐 API live at: {ngrok_tunnel.public_url}")
print(f"📖 Docs: {ngrok_tunnel.public_url}/docs")

thread = threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000))
thread.daemon = True
thread.start()
print("✅ Server running!")